# Antenna simulation
## Toolchain setup

In [ ]:
# === EMerge toolchain & Jupyter readiness check ===
# Run this cell first. Every line should read [OK]. Any [FAIL] tells you what to fix.
import sys
from importlib.metadata import version, PackageNotFoundError


def check(label, ok, detail=""):
    print(f"[{'OK  ' if ok else 'FAIL'}] {label}" + (f"  ->  {detail}" if detail else ""))
    return ok


print("=" * 62)
print(" EMerge toolchain / Jupyter readiness")
print("=" * 62)
ok_all = True

# 1) Python version -- EMerge requires >=3.10 and <3.14
v = sys.version_info
py_ok = (3, 10) <= (v.major, v.minor) <= (3, 13)
ok_all &= check(
    f"Python {v.major}.{v.minor}.{v.micro}",
    py_ok,
    "in 3.10-3.13" if py_ok else "EMerge needs 3.10-3.13 -> rebuild the venv with python 3.12",
)

# 2) Are we really inside the project venv (and not the system 3.14)?
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
check("Virtual environment active", in_venv, sys.prefix if in_venv else "SYSTEM python, no venv")
print(f"        interpreter: {sys.executable}")

# 3) EMerge itself
try:
    ok_all &= check("EMerge installed", True, f"v{version('emerge')}")
except PackageNotFoundError:
    ok_all &= check("EMerge installed", False, "missing -> uv pip install emerge")
try:
    import emerge  # noqa: F401
    check("import emerge", True)
except Exception as e:
    ok_all &= check("import emerge", False, f"{type(e).__name__}: {e}")

# 4) Core dependencies: import AND report version
print("-" * 62)
for mod in ["numpy", "scipy", "matplotlib", "gmsh", "pyvista", "joblib", "loguru"]:
    try:
        __import__(mod)
        ok_all &= check(mod, True, f"v{version(mod)}")
    except Exception as e:
        ok_all &= check(mod, False, f"{type(e).__name__}: {e}")

# 5) numpy must satisfy EMerge's pin: >=2.2,<2.5 (the 3.14 trap you hit earlier)
try:
    import numpy as np
    nv = tuple(int(x) for x in np.__version__.split(".")[:2])
    ok_all &= check(f"numpy {np.__version__} matches EMerge pin (>=2.2,<2.5)", (2, 2) <= nv < (2, 5))
except Exception as e:
    check("numpy range check", False, str(e))

# 6) Jupyter / kernel readiness
print("-" * 62)
try:
    check("ipykernel installed (env selectable as a kernel)", True, f"v{version('ipykernel')}")
except PackageNotFoundError:
    ok_all &= check("ipykernel installed", False, "uv pip install ipykernel")
try:
    check("running inside a Jupyter/IPython kernel", True, type(get_ipython()).__name__)  # noqa: F821
except NameError:
    check("running inside a Jupyter/IPython kernel", False, "not a notebook kernel")

print("=" * 62)
print(" RESULT:", "toolchain READY" if ok_all else "NOT ready -> fix the [FAIL] lines above")
print("=" * 62)
